# Data Preprocessing and Feature Extraction
[This notebook performs data preprocessing and feature extraction using a modified ConvNeXt model. The preprocessing includes various image enhancement techniques, and the features are extracted from MRI images of brain tumors.]

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import convnext_base, ConvNeXt_Base_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
from torchvision.models import densenet121, DenseNet121_Weights, vgg19, VGG19_Weights

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS device")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"MPS not available, using: {device}")

In [ ]:
DATASET_PATH = 'dataset'
TRAIN_PATH = os.path.join(DATASET_PATH, 'training')
TEST_PATH = os.path.join(DATASET_PATH, 'testing')
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']

model_name = "vgg19"

OUTPUT_PATH = f'1_Feature_Extraction/{model_name}'

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
class ImagePreprocessor:
    def __init__(self):
        pass
    
    def adjust_contrast(self, image, alpha=1.5):
        """Adjust contrast using alpha factor"""
        adjusted = cv2.convertScaleAbs(image, alpha=alpha, beta=0)
        return adjusted
    
    def apply_histogram_equalization(self, image):
        """Apply histogram equalization to enhance contrast"""
        return cv2.equalizeHist(image)
    
    def apply_clahe(self, image, clip_limit=2.0, tile_grid_size=(8, 8)):
        """Apply Contrast Limited Adaptive Histogram Equalization"""
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
        return clahe.apply(image)
    
    def apply_gaussian_blur(self, image, kernel_size=(5, 5)):
        """Apply Gaussian blur to reduce noise"""
        return cv2.GaussianBlur(image, kernel_size, 0)
    
    def apply_median_blur(self, image, kernel_size=5):
        """Apply median blur to reduce noise while preserving edges"""
        return cv2.medianBlur(image, kernel_size)
    
    def apply_bilateral_filter(self, image, d=9, sigma_color=75, sigma_space=75):
        """Apply bilateral filter to reduce noise while preserving edges"""
        return cv2.bilateralFilter(image, d, sigma_color, sigma_space)
    
    def apply_unsharp_mask(self, image, kernel_size=(5, 5), strength=1.5):
        """Apply unsharp mask to enhance edges"""
        blurred = cv2.GaussianBlur(image, kernel_size, 0)
        return cv2.addWeighted(image, 1.0 + strength, blurred, -strength, 0)
    
    def apply_sobel_filter(self, image):
        """Apply Sobel filter for edge detection"""
        sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
        sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
        sobel = np.sqrt(sobel_x**2 + sobel_y**2)
        sobel = np.uint8(sobel * 255.0 / np.max(sobel))
        return sobel
    
    def apply_canny_edge(self, image, threshold1=30, threshold2=100):
        """Apply Canny edge detection"""
        return cv2.Canny(image, threshold1, threshold2)
    
    def apply_otsu_thresholding(self, image):
        """Apply Otsu's thresholding"""
        _, thresh = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return thresh
    
    def apply_skull_stripping(self, image):
        """Simple skull stripping using thresholding and morphological operations"""
        # This is a simplified version - for actual skull stripping, use specialized libraries
        _, thresh = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        kernel = np.ones((5, 5), np.uint8)
        opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
        sure_bg = cv2.dilate(opening, kernel, iterations=3)
        
        # Apply the mask to the original image
        result = cv2.bitwise_and(image, image, mask=sure_bg)
        return result
    
    def normalize_image(self, image):
        """Normalize image to 0-1 range"""
        normalized = image.astype(np.float32) / 255.0
        return normalized
    
    def standardize_image(self, image):
        """Standardize image (zero mean, unit variance)"""
        if np.std(image) == 0:
            return np.zeros_like(image, dtype=np.float32)
        standardized = (image - np.mean(image)) / np.std(image)
        return standardized
    
    # New additional preprocessing methods
    def blend_with_edges(self, image, edge_weight=0.3):
        """Blend original image with its edge map to enhance structures"""
        edges = self.apply_canny_edge(image)
        return cv2.addWeighted(image, 1.0, edges, edge_weight, 0)
    
    def multi_scale_enhancement(self, image):
        """Enhance features at multiple scales using Gaussian pyramid"""
        # Create a Gaussian pyramid
        g0 = image.copy()
        g1 = cv2.pyrDown(g0)
        g2 = cv2.pyrDown(g1)
        
        # Expand back
        e1 = cv2.pyrUp(g1)
        e2 = cv2.pyrUp(g2)
        e2 = cv2.pyrUp(e2)
        
        # Resize to original size
        if e1.shape != g0.shape:
            e1 = cv2.resize(e1, (g0.shape[1], g0.shape[0]))
        if e2.shape != g0.shape:
            e2 = cv2.resize(e2, (g0.shape[1], g0.shape[0]))
        
        # Blend for multi-scale enhancement
        result = cv2.addWeighted(g0, 0.6, e1, 0.3, 0)
        result = cv2.addWeighted(result, 0.8, e2, 0.2, 0)
        
        return result
    
    def adaptive_thresholding(self, image, block_size=11, c=2):
        """Apply adaptive thresholding"""
        return cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                    cv2.THRESH_BINARY, block_size, c)
    
    def combine_preprocessing(self, image):
        """Combine multiple preprocessing techniques for best results"""
        # Apply CLAHE for contrast enhancement
        clahe_img = self.apply_clahe(image)
        
        # Apply median blur to reduce noise while preserving edges
        denoised = self.apply_median_blur(clahe_img, kernel_size=3)
        
        # Enhance edges
        edge_enhanced = self.blend_with_edges(denoised, edge_weight=0.2)
        
        # Apply unsharp masking for sharpening
        sharpened = self.apply_unsharp_mask(edge_enhanced)
        
        return sharpened
    
    def apply_all_filters(self, image):
        """Apply all filters and return a dictionary of processed images"""
        processed = {}
        processed['original'] = image
        processed['contrast_adjusted'] = self.adjust_contrast(image)
        processed['histogram_equalized'] = self.apply_histogram_equalization(image)
        processed['clahe'] = self.apply_clahe(image)
        processed['gaussian_blur'] = self.apply_gaussian_blur(image)
        processed['median_blur'] = self.apply_median_blur(image)
        processed['bilateral_filter'] = self.apply_bilateral_filter(image)
        processed['unsharp_mask'] = self.apply_unsharp_mask(image)
        processed['sobel_filter'] = self.apply_sobel_filter(image)
        processed['canny_edge'] = self.apply_canny_edge(image)
        processed['otsu_thresholding'] = self.apply_otsu_thresholding(image)
        processed['skull_stripped'] = self.apply_skull_stripping(image)
        processed['edge_blended'] = self.blend_with_edges(image)
        processed['multi_scale'] = self.multi_scale_enhancement(image)
        processed['adaptive_threshold'] = self.adaptive_thresholding(image)
        processed['combined'] = self.combine_preprocessing(image)
        return processed
    
    def visualize_filters(self, processed_images, save_path=None):
        """Visualize all the filters applied to an image"""
        # Calculate grid dimensions based on number of images
        num_images = len(processed_images)
        cols = 4  # 4 images per row
        rows = (num_images + cols - 1) // cols  # Ceiling division
        
        fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
        fig.tight_layout()
        
        # Flatten the axes array for easier indexing
        axes = axes.flatten() if num_images > cols else [axes]
        
        # Plot each processed image
        for i, (name, img) in enumerate(processed_images.items()):
            if i < rows * cols:
                axes[i].imshow(img, cmap='gray')
                axes[i].set_title(name)
                axes[i].axis('off')
        
        # Hide any unused subplots
        for i in range(len(processed_images), rows * cols):
            if i < len(axes):
                fig.delaxes(axes[i])
        
        plt.subplots_adjust(hspace=0.3, wspace=0.3)
        
        if save_path:
            plt.savefig(save_path, bbox_inches='tight', dpi=1000)
            plt.show()
            plt.close()
        else:
            plt.show()

In [ ]:
class MRIDataset(Dataset):
    def __init__(self, root_dir, classes, transform=None, preprocessor=None, preprocess_method=None):
        """
        Args:
            root_dir (string): Directory with all the images
            classes (list): List of class folder names
            transform (callable, optional): Transform to be applied on a sample
            preprocessor (ImagePreprocessor, optional): Preprocessor object
            preprocess_method (string, optional): Name of preprocessing method to apply
        """
        self.root_dir = root_dir
        self.classes = classes
        self.transform = transform
        self.preprocessor = preprocessor
        self.preprocess_method = preprocess_method
        
        self.image_paths = []
        self.labels = []
        self.file_names = []  # Store filenames for tracking
        
        # Load all image paths and labels
        for class_idx, class_name in enumerate(classes):
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.exists(class_dir):
                print(f"Warning: {class_dir} does not exist.")
                continue
                
            for img_name in os.listdir(class_dir):
                if img_name.endswith(('.jpg', '.jpeg', '.png')):
                    self.image_paths.append(os.path.join(class_dir, img_name))
                    self.labels.append(class_idx)
                    self.file_names.append(img_name)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        file_name = self.file_names[idx]
        
        # Read image in grayscale
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        # Apply preprocessing if specified
        if self.preprocessor and self.preprocess_method:
            preprocess_func = getattr(self.preprocessor, self.preprocess_method, None)
            if preprocess_func:
                image = preprocess_func(image)
        
        # Convert to RGB (3 channels) for model compatibility
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        
        # Convert to PIL Image for PyTorch transforms
        image = Image.fromarray(image)
        
        if self.transform:
            image = self.transform(image)
        
        return image, label, file_name

In [ ]:
def get_transforms():
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, test_transform

# Modified ConvNeXt model

In [ ]:
class ModifiedConvNeXt(nn.Module):
    def __init__(self):
        super(ModifiedConvNeXt, self).__init__()
        self.base = convnext_base(weights=ConvNeXt_Base_Weights.IMAGENET1K_V1)
        self.base.classifier[2] = nn.Identity()  # Remove original classifier
        self.custom_fc = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1024)  # Output 1024 features
        )
    
    def forward(self, x):
        features = self.base(x)
        features = self.custom_fc(features)
        return features, features  # Return features twice for compatibility with extract_features function

# Modified DenseNet121

In [ ]:
class ModifiedDenseNet121(nn.Module):
    def __init__(self):
        super(ModifiedDenseNet121, self).__init__()
        self.base = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        self.base.classifier = nn.Identity()  # Remove original classifier
        self.custom_fc = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1024)  # Output 1024 features
        )
    
    def forward(self, x):
        features = self.base(x)
        features = self.custom_fc(features)
        return features, features  # Return features twice for compatibility with extract_features function

# Modified VGG19

In [ ]:
class ModifiedVGG19(nn.Module):
    def __init__(self):
        super(ModifiedVGG19, self).__init__()
        self.base = vgg19(weights=VGG19_Weights.IMAGENET1K_V1)
        # Remove the original classifier
        self.features = self.base.features
        # The output of VGG19 features is 512 channels
        self.avgpool = self.base.avgpool
        
        # Replace classifier with custom FC layers
        self.custom_fc = nn.Sequential(
            nn.Linear(512 * 7 * 7, 512),  # VGG19 produces 512 feature maps of size 7x7 after pooling
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1024)  # Output 1024 features to match other models
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)  # Flatten the feature maps
        features = self.custom_fc(x)
        return features, features

In [ ]:
def extract_features(model, dataloader, device):
    model.eval()
    all_features = []
    all_labels = []
    all_file_names = []
    
    with torch.no_grad():
        for inputs, labels, file_names in tqdm(dataloader, desc="Extracting features"):
            inputs = inputs.to(device)
            _, features = model(inputs)
            
            # Move features back to CPU for storage
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
            all_file_names.extend(file_names)
    
    # Concatenate all features and labels
    all_features = np.vstack(all_features)
    all_labels = np.concatenate(all_labels)
    
    return all_features, all_labels, all_file_names

In [ ]:
def save_features_to_csv(features, labels, file_names, output_path, filename, class_names):
    # Create feature column names
    feature_cols = [f"feat_{i}" for i in range(features.shape[1])]
    
    # Convert numeric labels to class names
    label_names = [class_names[label] for label in labels]
    
    # Create a DataFrame with features and label
    df = pd.DataFrame(features, columns=feature_cols)
    df['label'] = label_names
    df['file_name'] = file_names
    
    # Save to CSV
    os.makedirs(output_path, exist_ok=True)
    output_file = os.path.join(output_path, filename)
    df.to_csv(output_file, index=False)
    print(f"Saved features to {output_file}")

In [ ]:
def demonstrate_preprocessing():
    # Sample image paths (adjust these to match your dataset structure)
    sample_images = [
        os.path.join(TEST_PATH, 'glioma', 'Te-gl_0010.jpg'),
        os.path.join(TEST_PATH, 'meningioma', 'Te-me_0010.jpg'),
        os.path.join(TEST_PATH, 'notumor', 'Te-no_0010.jpg'),
        os.path.join(TEST_PATH, 'pituitary', 'Te-pi_0010.jpg')
    ]
    
    preprocessor = ImagePreprocessor()
    
    for img_path in sample_images:
        if os.path.exists(img_path):
            print(f"Processing: {img_path}")
            
            # Read the image
            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            
            # Apply all filters
            processed_images = preprocessor.apply_all_filters(image)
            
            # Create output directory for visualizations
            vis_dir = os.path.join(OUTPUT_PATH, 'visualizations')
            os.makedirs(vis_dir, exist_ok=True)
            
            # Get filename without extension
            filename = os.path.splitext(os.path.basename(img_path))[0]
            save_path_png = os.path.join(vis_dir, f"{filename}_filters.png")
            save_path_pdf = os.path.join(vis_dir, f"{filename}_filters.pdf")
            
            # Visualize and save
            preprocessor.visualize_filters(processed_images, save_path_png)
            preprocessor.visualize_filters(processed_images, save_path_pdf)
            print(f"Saved filter visualization to {save_path_pdf} & {save_path_png}")
        else:
            print(f"Image not found: {img_path}")

In [ ]:
def compare_preprocessing_methods():
    # Create visualization path
    vis_dir = os.path.join(OUTPUT_PATH, 'visualizations')
    os.makedirs(vis_dir, exist_ok=True)
    
    # Sample images for each class
    sample_images = {
        "glioma": "Te-gl_0010.jpg",
        "meningioma": "Te-me_0010.jpg",
        "notumor": "Te-no_0010.jpg",
        "pituitary": "Te-pi_0010.jpg",
    }
    
    preprocessor = ImagePreprocessor()
    
    plt.figure(figsize=(15, 12))
    
    for i, (cls, img_name) in enumerate(sample_images.items()):
        img_path = os.path.join(TEST_PATH, cls, img_name)
        if not os.path.exists(img_path):
            print(f"Image not found: {img_path}")
            continue
        
        # Read image
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        # Apply different preprocessing methods
        original = image
        clahe = preprocessor.apply_clahe(image)
        edge_enhanced = preprocessor.blend_with_edges(image)
        combined = preprocessor.combine_preprocessing(image)
        
        # Plot
        plt.subplot(4, 4, i*4 + 1)
        plt.imshow(original, cmap='gray')
        plt.title(f"{cls} - Original")
        plt.axis("off")
        
        plt.subplot(4, 4, i*4 + 2)
        plt.imshow(clahe, cmap='gray')
        plt.title(f"{cls} - CLAHE")
        plt.axis("off")
        
        plt.subplot(4, 4, i*4 + 3)
        plt.imshow(edge_enhanced, cmap='gray')
        plt.title(f"{cls} - Edge Enhanced")
        plt.axis("off")
        
        plt.subplot(4, 4, i*4 + 4)
        plt.imshow(combined, cmap='gray')
        plt.title(f"{cls} - Combined")
        plt.axis("off")
    
    plt.tight_layout()
    save_path_png = os.path.join(vis_dir, "preprocessing_comparison.png")
    save_path_pdf = os.path.join(vis_dir, "preprocessing_comparison.png")
    plt.savefig(save_path_png, bbox_inches='tight', dpi=1000)
    plt.savefig(save_path_pdf, bbox_inches='tight', dpi=1000)
    plt.show()
    plt.close()
    print(f"Saved preprocessing comparison to {save_path_pdf} & {save_path_png}")

In [ ]:
def visualize_filters_from_second_file():
    # Sample images from the second file
    sample_images = {
        "testing/glioma": "Te-gl_0010.jpg",
        "testing/meningioma": "Te-me_0010.jpg",
        "testing/notumor": "Te-no_0010.jpg",
        "testing/pituitary": "Te-pi_0010.jpg"
    }
    
    # Create visualization path
    vis_dir = os.path.join(OUTPUT_PATH, 'visualizations')
    os.makedirs(vis_dir, exist_ok=True)
    
    # Preprocessing transforms from the second file
    preprocess = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    plt.figure(figsize=(15, 10))
    for i, (key, img_name) in enumerate(sample_images.items()):
        img_path = os.path.join(DATASET_PATH, key, img_name)
        if not os.path.exists(img_path):
            print(f"Image not found: {img_path}")
            continue
        
        img = Image.open(img_path).convert("RGB")
        img_tensor = preprocess(img).unsqueeze(0)  # Add batch dimension
        
        # Apply filters (from paste-2.txt)
        img_np = img_tensor[0].numpy().transpose(1, 2, 0)  # Convert to HWC
        img_np = (img_np * 255).astype(np.uint8)  # Denormalize to 0-255
        gaussian = cv2.GaussianBlur(img_np, (5, 5), 0)
        edges = cv2.Canny(img_np, 100, 200)
        
        # Plot
        plt.subplot(len(sample_images), 3, i * 3 + 1)
        plt.imshow(img_np)
        plt.title(f"{key} - Original")
        plt.axis("off")
        
        plt.subplot(len(sample_images), 3, i * 3 + 2)
        plt.imshow(gaussian)
        plt.title(f"{key} - Gaussian")
        plt.axis("off")
        
        plt.subplot(len(sample_images), 3, i * 3 + 3)
        plt.imshow(edges, cmap="gray")
        plt.title(f"{key} - Edges")
        plt.axis("off")
    
    plt.tight_layout()
    save_path_png = os.path.join(vis_dir, "filters_visualization_second_file.png")
    save_path_pdf = os.path.join(vis_dir, "filters_visualization_second_file.png")
    plt.savefig(save_path_pdf, bbox_inches='tight', dpi=1000)
    plt.savefig(save_path_png, bbox_inches='tight', dpi=1000)
    plt.show()
    plt.close()
    print(f"Saved filters visualization from second file to {save_path_pdf} & {save_path_png}")

In [ ]:
demonstrate_preprocessing()

In [ ]:
visualize_filters_from_second_file()

In [ ]:
preprocessor = ImagePreprocessor()

In [ ]:
train_transform, test_transform = get_transforms()

# Check if the paths exist
print(f"Train path exists: {os.path.exists(TRAIN_PATH)}")
print(f"Test path exists: {os.path.exists(TEST_PATH)}")

In [ ]:
train_dataset = MRIDataset(
    TRAIN_PATH, 
    CLASSES, 
    transform=train_transform, 
    preprocessor=preprocessor, 
    preprocess_method='combine_preprocessing'
)

test_dataset = MRIDataset(
    TEST_PATH, 
    CLASSES, 
    transform=test_transform, 
    preprocessor=preprocessor, 
    preprocess_method='combine_preprocessing'
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

In [ ]:
model = ModifiedVGG19()
model = model.to(device)

# Save the model architecture to a file
model_path = os.path.join(OUTPUT_PATH, f'modified_{model_name}_model.pth')
torch.save(model.state_dict(), model_path)

# Save the model architecture into a text file
model_architecture_path = os.path.join(OUTPUT_PATH, f'modified_{model_name}_model_architecture.txt')
with open(model_architecture_path, 'w') as f:
    f.write(str(model))

In [ ]:
print("Extracting features from training set...")
train_features, train_labels, train_file_names = extract_features(model, train_dataloader, device)

In [ ]:
print("Extracting features from testing set...")
test_features, test_labels, test_file_names = extract_features(model, test_dataloader, device)

In [ ]:
# Save features to CSV
save_features_to_csv(train_features, train_labels, train_file_names, OUTPUT_PATH, f'train_features_{model_name}.csv', CLASSES)
save_features_to_csv(test_features, test_labels, test_file_names, OUTPUT_PATH, f'test_features_{model_name}.csv', CLASSES)

In [ ]:
print("Feature extraction complete!")